In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import os

In [ ]:
os.listdir('images/train')

In [ ]:
label = {
    'NORMAL': 0,
    'PNEUMONIA': 1
}

In [ ]:
X_train = []
y_train = []

for i in os.listdir('images/train'):
    for j in os.listdir(f'images/train/{i}'):
        image = cv2.imread(f'images/train/{i}/{j}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image,(100,100))
        image = image / 255
        image = image.reshape(100,100,1)
        X_train.append(image)
        y_train.append(label[i])

X_train = np.array(X_train)
y_train = np.array(y_train)

In [ ]:
X_test = []
y_test = []

for i in os.listdir('images/test'):
    for j in os.listdir(f'images/test/{i}'):
        image = cv2.imread(f'images/test/{i}/{j}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image,(100,100))
        image = image / 255
        image = image.reshape(100,100,1)
        X_test.append(image)
        y_test.append(label[i])

X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
from sklearn.utils import shuffle
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
X_train, y_train = shuffle(X_train, y_train)
X_test, y_test = shuffle(X_test, y_test)

In [ ]:
from keras.utils import to_categorical

In [ ]:
class_weights = compute_class_weight(class_weight='balanced',classes=np.unique(y_train),y=y_train)
class_weights = dict(enumerate(class_weights))

y_train = to_categorical(y_train,num_classes=2)
y_test = to_categorical(y_test,num_classes=2)

In [ ]:
from keras import Sequential
from keras.layers import Conv2D,Dense,MaxPooling2D,Flatten
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

In [ ]:
model.add(Conv2D(filters=100,kernel_size=(5,5),input_shape=(100,100,1),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(filters=100,kernel_size=(3,3),activation='relu'))
model.add(Conv2D(filters=100,kernel_size=(3,3),activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(units=32,activation='relu'))
model.add(Dense(units=16,activation='relu'))
model.add(Dense(units=8,activation='relu'))
model.add(Dense(units=4,activation='relu'))
model.add(Dense(units=2,activation='softmax'))


In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001),loss='categorical_crossentropy',metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)

In [ ]:
model.fit(X_train,y_train,epochs=7,validation_data=(X_test, y_test),class_weight=class_weights,callbacks=[early_stop])

In [ ]:
test_image = mpimg.imread('images.jpg')

In [ ]:
plt.imshow(test_image)

In [ ]:
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2GRAY)

In [ ]:
test_image = cv2.resize(test_image, (100, 100))

In [ ]:
test_image = test_image / 255

In [ ]:
test_image = test_image.reshape(1, 100, 100, 1)

In [ ]:
model.predict(test_image)